<a href="https://colab.research.google.com/github/xujiahengbill-bot/Harris/blob/main/student_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

# Load data
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")


Step 2: Split Features & Target

In [24]:
train_df.head()
test_df.head()

,id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty
0,630000,24,other,ba,6.85,65.2,yes,5.2,poor,group study,high,easy
1,630001,18,male,diploma,6.61,45.0,no,9.3,poor,coaching,low,easy
2,630002,24,female,b.tech,6.60,98.5,yes,6.2,good,group study,medium,moderate
3,630003,24,male,diploma,3.03,66.3,yes,5.7,average,mixed,medium,moderate
4,630004,20,female,b.tech,2.03,42.4,yes,9.2,average,coaching,low,moderate


In [3]:
TARGET = "exam_score"

X = train_df.drop(columns=[TARGET])
y = train_df[TARGET]

X_test = test_df.copy()


Step 3: Define All Models

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

results = []

# Define categorical and numerical features
# 'id' column should be excluded from features
numerical_features = ['age', 'study_hours', 'class_attendance', 'sleep_hours']
categorical_features = [
    'gender', 'course', 'internet_access', 'sleep_quality',
    'study_method', 'facility_rating', 'exam_difficulty'
]

# Create a preprocessor using ColumnTransformer
preprocess = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough' # This will pass through any other columns not specified (like 'id' if not dropped)
)

In [5]:
def run_linear_regression():
    pipe = Pipeline([("preprocess", preprocess), ("model", LinearRegression())])
    return evaluate_pipeline(pipe, "LinearRegression")

def run_ridge():
    pipe = Pipeline([("preprocess", preprocess), ("model", Ridge(alpha=1.0, random_state=42))])
    return evaluate_pipeline(pipe, "Ridge")

def run_lasso():
    pipe = Pipeline([("preprocess", preprocess), ("model", Lasso(alpha=0.001, random_state=42, max_iter=50000))])
    return evaluate_pipeline(pipe, "Lasso")

def run_elasticnet():
    pipe = Pipeline([("preprocess", preprocess), ("model", ElasticNet(alpha=0.001, l1_ratio=0.5, random_state=42, max_iter=50000))])
    return evaluate_pipeline(pipe, "ElasticNet")

def run_sgd():
    pipe = Pipeline([("preprocess", preprocess), ("model", SGDRegressor(random_state=42, max_iter=5000, tol=1e-3))])
    return evaluate_pipeline(pipe, "SGDRegressor")

def run_decision_tree():
    pipe = Pipeline([("preprocess", preprocess), ("model", DecisionTreeRegressor(random_state=42))])
    return evaluate_pipeline(pipe, "DecisionTree")

def run_random_forest():
    pipe = Pipeline([("preprocess", preprocess), ("model", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
    return evaluate_pipeline(pipe, "RandomForest")

def run_extra_trees():
    pipe = Pipeline([("preprocess", preprocess), ("model", ExtraTreesRegressor(n_estimators=100, random_state=42, n_jobs=-1))])
    return evaluate_pipeline(pipe, "ExtraTrees")

def run_gradient_boosting():
    pipe = Pipeline([("preprocess", preprocess), ("model", GradientBoostingRegressor(random_state=42))])
    return evaluate_pipeline(pipe, "GradientBoosting")

def run_hist_gradient_boosting():
    pipe = Pipeline([("preprocess", preprocess), ("model", HistGradientBoostingRegressor(random_state=42))])
    return evaluate_pipeline(pipe, "HistGradientBoosting")

In [6]:
def evaluate_pipeline(pipe, model_name):
    scores = cross_val_score(
        pipe, X, y,
        cv=5,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1
    )
    rmse = -scores.mean()
    std = scores.std()
    print(f"{model_name:22s} | CV RMSE: {rmse:.5f} | std: {std:.5f}")
    return rmse

In [7]:
rmse_linear = run_linear_regression()
rmse_ridge  = run_ridge()
rmse_lasso  = run_lasso()


LinearRegression       | CV RMSE: 8.89478 | std: 0.01087
Ridge                  | CV RMSE: 8.89478 | std: 0.01087
Lasso                  | CV RMSE: 8.89477 | std: 0.01089


In [8]:
rmse_enet   = run_elasticnet()
rmse_sgd    = run_sgd()



ElasticNet             | CV RMSE: 8.89478 | std: 0.01090
SGDRegressor           | CV RMSE: 7346078311569254400.00000 | std: 4617615110375042048.00000


In [9]:
rmse_dt     = run_decision_tree()




DecisionTree           | CV RMSE: 13.14701 | std: 0.50835


TerminatedWorkerError: A worker process managed by the executor was unexpectedly terminated. This could be caused by a segmentation fault while calling the function or by an excessive memory usage causing the Operating System to kill the worker.

The exit codes of the workers are {SIGKILL(-9)}
Detailed tracebacks of the workers should have been printed to stderr in the executor process if faulthandler was not disabled.

In [16]:
lasso_model = Pipeline([
    ("preprocess", preprocess),
    ("model", Lasso(alpha=0.001, max_iter=50000, random_state=42))
])

lasso_model.fit(X, y)


/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocess',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['age', 'study_hours',
                                                   'class_attendance',
                                                   'sleep_hours']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['gender', 'course',
                                                   'internet_access',
                                                   'sleep_quality',
                                                   'study_method',
                                                   'facility_rating',
                                                   'exam_difficulty'])])),
                ('model', Lasso(alpha=0.001, max_iter=50000, random_state=42))])

In [27]:
test_pred = lasso_model.predict(X_test)
test_pred_rounded = test_pred.round(1)
test_pred_rounded


array([71.8, 69.5, 87.4, ..., 90. , 55.5, 68.4])

In [29]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "exam_score": test_pred_rounded
})

submission.to_csv("submission.csv", index=False)
submission.head()


,id,exam_score
0,630000,71.8
1,630001,69.5
2,630002,87.4
3,630003,54.9
4,630004,47.3
